# Model training - Implicit ALS and results

This notebook trains the ALS model and evaluates it with the same leave-one-out setup defined in `main.py`. The EDA and task reasoning are in `analysis.ipynb`.

---

## 1. Problem framing

The final task is top-K chapter recommendation with implicit feedback. Since there are no ratings or timestamps, this is not a true next-item prediction problem. For each user, we rank the held-out chapter against all catalog chapters that were not in that user's training set.

---

## 2. Data summary

From `analysis.ipynb`, about 99.96% of `(user, book)` pairs contain only one distinct chapter. So the interaction data does not provide a reliable within-book sequence. Because the data is also very sparse, global popularity is an important baseline.

---

## 3. Evaluation protocol

We use leave-one-out per user: hold out one chapter for users with at least 2 chapters, and keep the rest in train. Candidates are all catalog chapters except the user's training chapters, so the held-out chapter is still a valid candidate. Metrics used here are Hit@K, MRR, and NDCG@K.

---

## 4. Model

The main model is implicit ALS. Users and chapters get learned vectors, and the score is their dot product. The training matrix must stay in `users x chapters` format, and raw counts are passed into `fit()`.

---

## 5. Code flow

The notebook loads data, builds the leave-one-out split, creates the sparse training matrix, fits ALS, runs a quick sanity check for one user, and then compares ALS with the popularity baselines.

---

## 6-7. Results and write-up

Run the notebook through the comparison table, then read the final notes below for the results, limitations, and conclusion.


### Setup

Run this once in a terminal: `python3 -m venv .venv && .venv/bin/pip install -r requirements-training.txt`. After that, choose the `.venv` kernel in Jupyter and run the notebook from top to bottom.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse as sp

try:
    from implicit.als import AlternatingLeastSquares
except ImportError as e:
    raise ImportError(
        "Install implicit in a venv: pip install -r requirements-training.txt"
    ) from e

from main import (
    all_chapter_ids,
    build_leave_one_out_splits,
    build_primary_tag_train_counts,
    chapter_index_map,
    metrics_from_ranks,
    popularity_scores_aligned,
    ranks_from_global_scores,
    ranks_from_user_item_factors,
    subsample_splits,
    tag_scores_aligned,
    train_chapter_counts,
)

DATA = Path(".")
SEED = 42
# Subsample eval users only (full train matrix still used for ALS unless you change construction)
SUBSAMPLE_EVAL_USERS = None  # e.g. 20_000 for faster rank pass



/opt/homebrew/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5a. Load data and build the leave-one-out split

This cell loads both files and creates `splits` and `train_ix` using `main.build_leave_one_out_splits`.


In [2]:
chapters = pd.read_csv(DATA / "chapters.csv")
interactions = pd.read_csv(DATA / "interactions.csv")

splits, train_ix = build_leave_one_out_splits(interactions, seed=SEED)
if SUBSAMPLE_EVAL_USERS is not None:
    splits = subsample_splits(splits, SUBSAMPLE_EVAL_USERS, seed=SEED)

chapter_ids = all_chapter_ids(chapters)
c2i = chapter_index_map(chapter_ids)
n_items = len(chapter_ids)

print("Eval users:", f"{len(splits):,}")
print("Train interaction rows:", f"{len(train_ix):,}")
print("Catalog items (matrix columns):", n_items)

Eval users: 148,549
Train interaction rows: 851,451
Catalog items (matrix columns): 50000


## 5b. Training matrix (CSR)

Rows are users from `train_ix`, columns are all catalog chapters in a fixed order, and the values are training interaction counts.


In [3]:
agg = train_ix.groupby(["user_id", "chapter_id"], sort=False).size().reset_index(name="cnt")

unique_users = agg["user_id"].unique()
user_to_row = {u: i for i, u in enumerate(unique_users)}
n_users = len(unique_users)

rows = agg["user_id"].map(user_to_row).to_numpy(dtype=np.int32)
cols = agg["chapter_id"].map(lambda x: c2i[int(x)]).to_numpy(dtype=np.int32)
data = agg["cnt"].to_numpy(dtype=np.float32)

train_matrix = sp.csr_matrix((data, (rows, cols)), shape=(n_users, n_items), dtype=np.float32)
train_matrix.eliminate_zeros()
print("CSR shape:", train_matrix.shape, "nnz:", train_matrix.nnz)

CSR shape: (149803, 50000) nnz: 851451


In [4]:
# Assertions: CSR orientation matches implicit's fit(user_items) and factor shapes
assert train_matrix.shape == (n_users, n_items), (train_matrix.shape, (n_users, n_items))
assert train_matrix.shape[0] == len(unique_users)
assert train_matrix.shape[1] == n_items == len(chapter_ids)
print(
    "OK: train_matrix is (n_users, n_chapters). "
    "After fit, expect user_factors (n_users, k) and item_factors (n_chapters, k)."
)

OK: train_matrix is (n_users, n_chapters). After fit, expect user_factors (n_users, k) and item_factors (n_chapters, k).


## 5c. Fit implicit ALS

The next cell sets the hyperparameters and trains the model. After `fit()`, the factor shapes should match `(n_users, k)` and `(n_chapters, k)`.


In [5]:
# Requires train_matrix from previous cells (unique_users, user_to_row, rows/cols/data).
ALS_FACTORS = 32  # try 32 vs 64; smaller often helps sparse data
ALS_ITERATIONS = 20
ALS_REG = 0.1  # from light grid; compare to 0.01 / 0.08 in als_tune.py
ALS_ALPHA = 40.0  # implicit scales Cui *= alpha inside fit — pass raw counts into fit()

model = AlternatingLeastSquares(
    factors=ALS_FACTORS,
    iterations=ALS_ITERATIONS,
    regularization=ALS_REG,
    alpha=ALS_ALPHA,
    random_state=SEED,
)
model.fit(train_matrix)

assert model.user_factors.shape == (n_users, ALS_FACTORS)
assert model.item_factors.shape == (n_items, ALS_FACTORS)
print("Factors OK:", model.user_factors.shape, model.item_factors.shape)


100%|████████████████████████████████████████████████| 20/20 [00:06<00:00,  2.99it/s]

Factors OK: (149803, 32) (50000, 32)


In [6]:
# One user: train masked, rank held-out among candidates
_split = splits[int(SEED) % len(splits)]
_uid, _urow = _split.user_id, user_to_row[_split.user_id]
_sc = model.user_factors[_urow] @ model.item_factors.T
_mask = np.zeros(n_items, dtype=bool)
for _c in _split.train_chapters:
    _mask[c2i[int(_c)]] = True
assert not _mask[c2i[_split.test_chapter]], "held-out must not be in train_chapters mask"
_sc_masked = _sc.copy()
_sc_masked[_mask] = -np.inf
_top5 = np.argsort(-_sc_masked)[:5]
_cids = chapter_ids.astype(np.int64)
_tid = c2i[_split.test_chapter]
_s_t, _c_t = float(_sc[_tid]), int(_cids[_tid])
_cand = ~_mask
_better = _cand & ((_sc > _s_t) | ((_sc == _s_t) & (_cids < _c_t)))
_rank = int(_better.sum() + 1)
print("Sanity user:", _uid)
print("  ALS top-5 chapter_ids (train masked):", chapter_ids[_top5].tolist())
print("  held-out chapter_id:", _split.test_chapter, "| rank among candidates:", _rank)

Sanity user: user_9305680
  ALS top-5 chapter_ids (train masked): [7332858, 9816279, 8309621, 3011388, 5875546]
  held-out chapter_id: 5714200 | rank among candidates: 31788


## 6. Results

This section compares ALS with global train popularity and primary-tag popularity using the same leave-one-out candidates and metrics.


In [ ]:
missing = [sp.user_id for sp in splits if sp.user_id not in user_to_row]
if missing:
    raise RuntimeError(f"{len(missing)} split users missing from train matrix")

user_row_indices = np.array([user_to_row[sp.user_id] for sp in splits], dtype=np.int32)

ranks_als = ranks_from_user_item_factors(
    splits,
    chapter_ids,
    c2i,
    user_row_indices,
    model.user_factors,
    model.item_factors,
    batch_size=512,
)
metrics_als = metrics_from_ranks(ranks_als)

train_counts = train_chapter_counts(train_ix)
pop_scores = popularity_scores_aligned(chapter_ids, train_counts)
ranks_pop = ranks_from_global_scores(splits, chapter_ids, pop_scores, c2i)
metrics_pop = metrics_from_ranks(ranks_pop)

tag_train = build_primary_tag_train_counts(train_ix, chapters)
tag_scores = tag_scores_aligned(chapter_ids, chapters, tag_train, train_counts)
ranks_tag = ranks_from_global_scores(splits, chapter_ids, tag_scores, c2i)
metrics_tag = metrics_from_ranks(ranks_tag)

comparison = pd.DataFrame(
    [metrics_als, metrics_pop, metrics_tag],
    index=["implicit_als", "global_train_popularity", "primary_tag_popularity"],
)
display(comparison.round(6))


## Results, limitations, and conclusion

**Interpretation:** Global popularity matches or slightly beats ALS on MRR, Hit@K, and NDCG@K. That makes sense for this dataset because the interactions are very sparse, users usually do not read multiple chapters from the same book, and there is no time information. So ALS can learn only limited personalization, while globally popular chapters stay a strong baseline.

I also tried a few simple metadata-based ideas, like a small ALS plus genre or author boosts and tag-cosine scoring. They did not improve the main comparison, so I left them out of the final table to keep the results focused.

**Limitations:** Without timestamps, leave-one-out is only a proxy and not a true next-in-time evaluation. There is also a cold-start issue for new users or chapters. Since each user has only one held-out positive among about 50k candidates, absolute Hit@K values are naturally very small, so relative comparison matters more.

**Takeaway:** This dataset is better suited for broad implicit recommendation than for sequence modeling. Popularity is a meaningful baseline here, and ALS is still a reasonable collaborative filtering reference.
